In [128]:
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

In [129]:
### Cell 2 — Imports
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import lightgbm as lgb
import shap
from sklearn.metrics import mean_squared_error
sns.set(style="darkgrid")

In [130]:
### Cell 3 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [131]:
### Cell 4 — Load Data
def read_file(filename):
    directory = '/content/drive/My Drive/210_capstone/final_datasets/full_hist_attr/'
    return pd.read_parquet(directory + filename + '.parquet')

In [132]:
df = read_file('full_data_18to23_asofFeb25_1')

In [133]:
### Cell 5 — Rename Columns
df = df.rename(columns={
    'Date':  'date',
    'County': 'county',
    'BEV':   'bev',
    'PHEV':  'phev',
    'FCEV':  'fcev',
    'Max_Daily_Electricity_Usage': 'max_daily_electricity',
    'Per_Capita_Personal_Income_Adjusted': 'per_cap_income',
    'Total': 'total_ev_charge'
})

## FEATURE ENGINEERING

In [134]:
### Cell 6 — Target Engineering (BEFORE split)
df['date'] = pd.to_datetime(df['date'])
df['max_elec_per_capita']     = df['max_daily_electricity'] / df['total_pop']
df['max_elec_per_capita_log'] = np.log(df['max_elec_per_capita'])

target = 'max_elec_per_capita_log'

print(df.shape)


(127078, 74)


In [135]:
# Monthly baseline from 2018
monthly_baseline_2018 = (df[df['date'].dt.year == 2018]
    .assign(month=lambda x: x['date'].dt.month)
    .groupby(['county', 'month'])
    .apply(lambda x: np.average(x['max_daily_electricity'], weights=x['total_pop']),
           include_groups=False)
    .reset_index()
    .rename(columns={0: 'baseline_mw_monthly_2018'}))

print(monthly_baseline_2018.head(20).to_string())

# Check statewide pop-weighted monthly baseline
statewide = (df[df['date'].dt.year == 2018]
    .assign(month=lambda x: x['date'].dt.month)
    .groupby('month')
    .apply(lambda x: np.average(x['max_daily_electricity'], weights=x['total_pop']),
           include_groups=False)
    .reset_index()
    .rename(columns={0: 'statewide_baseline_mw'}))

print("\nStatewide pop-weighted monthly baseline 2018:")
print(statewide.to_string(index=False))

     county  month  baseline_mw_monthly_2018
0   Alameda      1               1409.050922
1   Alameda      2               1435.616411
2   Alameda      3               1448.586942
3   Alameda      4               1410.874330
4   Alameda      5               1458.798573
5   Alameda      6               1624.530957
6   Alameda      7               1853.098624
7   Alameda      8               1741.385606
8   Alameda      9               1571.093310
9   Alameda     10               1408.985096
10  Alameda     11               1379.494960
11  Alameda     12               1374.451028
12   Alpine      1                  0.975642
13   Alpine      2                  1.068054
14   Alpine      3                  1.068804
15   Alpine      4                  1.061018
16   Alpine      5                  1.177404
17   Alpine      6                  1.320984
18   Alpine      7                  1.495994
19   Alpine      8                  1.391024

Statewide pop-weighted monthly baseline 2018:
 month  

In [136]:
ROLLING_SPECS = [
    ("cdd65_pop_roll5",       "cdd65_pop",  5,  "sum"),
    ("hdd65_pop_roll5",       "hdd65_pop",  5,  "sum"),
    ("tmax_k_pop_roll5_max",  "tmax_k_pop", 5,  "max"),
    ("tmax_k_pop_roll7_mean", "tmax_k_pop", 7,  "mean"),

]

def add_rolling_features(df):
    df = df.sort_values(["county", "date"]).copy()
    g = df.groupby("county", sort=False)
    for new_col, src_col, window, agg in ROLLING_SPECS:
        df[new_col] = g[src_col].transform(
            lambda x, w=window, a=agg: getattr(x.rolling(w, min_periods=1), a)()
        )
    return df

In [137]:
df = add_rolling_features(df)

In [138]:
df.columns

Index(['date', 'county', 'dpt_afternoon_k_pop', 'hdd65', 'wind_peak_ms_mean',
       'wind_low_ms_mean', 'spfh_peak_kgkg_pop', 'wind_peak_ms_pop', 'cdd75',
       'tavg_k', 'dpt_afternoon_k_mean', 'cloud_cover_pct_pop',
       'dpt_morning_k_mean', 'tmax_k_pop', 'cdd65_pop', 'cloud_cover_pct_mean',
       'tmin_k', 'hdd65_pop', 'dpt_morning_k_pop', 'wind_low_ms_pop',
       'trange_k', 'cdd65', 'tmin_k_pop', 'cdd75_pop', 'spfh_peak_kgkg_mean',
       'tmax_k', 'real_data_urma', 'staying_total', 'entering_total',
       'leaving_total', 'real_data_commuting', 'cuml_count', 'cuml_sq_foot',
       'cuml_utility_cap', 'cuml_dc_load', 'real_data_data_centers',
       'Total_Daily_Electricity_Usage', 'hour_of_max', 'max_daily_electricity',
       'Public Level 1', 'Shared Private Level 1', 'Public Level 2',
       'Shared Private Level 2', 'Public DC Fast', 'Shared Private DC Fast',
       'total_ev_charge', 'real_data_ev_charging', 'bev', 'phev', 'fcev',
       'real_data_ev_poplution', 'pe

In [139]:
### Dew Point Depression Feature
import numpy as np

def add_dpd_features(df):
    P = 101325  # standard pressure Pa
    w = df['spfh_peak_kgkg_pop']

    # specific humidity → vapor pressure
    e = (w * P) / (0.622 + w)

    # vapor pressure → dew point (K)
    dpt_c = 243.04 * np.log(e / 611.2) / (17.625 - np.log(e / 611.2))
    df['dpt_derived_k'] = dpt_c + 273.15

    # dew point depression (K) — large = dry, small = humid
    df['dpd_k'] = (df['tmax_k_pop'] - df['dpt_derived_k']).clip(lower=0)

    # rolling 5-day mean depression
    df = df.sort_values(['county', 'date'])
    df['dpd_k_roll5'] = (df
        .groupby('county')['dpd_k']
        .transform(lambda x: x.rolling(5, min_periods=1).mean()))

    return df

df = add_dpd_features(df)

#print(f"dpd_k stats:\n{df['dpd_k'].describe()}")

In [140]:
# Impute per_cap_income — county median, fall back to global median
global_median = df['per_cap_income'].median()

df['per_cap_income'] = (df.groupby('county')['per_cap_income']
    .transform(lambda x: x.fillna(x.median())))

df['per_cap_income'] = df['per_cap_income'].fillna(global_median)

print(df['per_cap_income'].isna().sum())  # should be 0

0


In [141]:
# Annual baseline — county level
baseline_2018 = (df[df['date'].dt.year == 2018]
    .groupby('county')
    .apply(lambda x: (x['max_daily_electricity'] / x['total_pop']).mean(),
           include_groups=False)
    .reset_index()
    .rename(columns={0: 'baseline_mw_per_capita_2018'}))

df = df.merge(baseline_2018, on='county', how='left')

# Monthly baseline — county x month level
monthly_baseline_2018 = (df[df['date'].dt.year == 2018]
    .assign(month=lambda x: x['date'].dt.month)
    .groupby(['county', 'month'])
    .apply(lambda x: np.average(x['max_daily_electricity'], weights=x['total_pop']),
           include_groups=False)
    .reset_index()
    .rename(columns={0: 'baseline_mw_monthly_2018'}))

df = df.assign(month=df['date'].dt.month).merge(
    monthly_baseline_2018, on=['county', 'month'], how='left')

print(df[['county', 'date', 'baseline_mw_per_capita_2018', 'baseline_mw_monthly_2018']].head(10))
print(f"\nNaNs: {df[['baseline_mw_per_capita_2018','baseline_mw_monthly_2018']].isna().sum().sum()}")

    county       date  baseline_mw_per_capita_2018  baseline_mw_monthly_2018
0  Alameda 2018-01-01                     0.000904               1409.050922
1  Alameda 2018-01-02                     0.000904               1409.050922
2  Alameda 2018-01-03                     0.000904               1409.050922
3  Alameda 2018-01-04                     0.000904               1409.050922
4  Alameda 2018-01-05                     0.000904               1409.050922
5  Alameda 2018-01-06                     0.000904               1409.050922
6  Alameda 2018-01-07                     0.000904               1409.050922
7  Alameda 2018-01-08                     0.000904               1409.050922
8  Alameda 2018-01-09                     0.000904               1409.050922
9  Alameda 2018-01-10                     0.000904               1409.050922

NaNs: 0


# THE SPLIT

In [142]:
# Run this before creating datasets — on df before the split
dow_map = {d: i for i, d in enumerate(df['day_of_week'].unique())}
df['day_of_week'] = df['day_of_week'].map(dow_map)

# Re-split
train_df = df[df['date'].dt.year <= 2021].copy()
val_df   = df[df['date'].dt.year == 2022].copy()
test_df  = df[df['date'].dt.year == 2023].copy()

In [143]:
# Split
train_df = df[df['date'].dt.year <= 2021].copy()
val_df   = df[df['date'].dt.year == 2022].copy()
test_df  = df[df['date'].dt.year == 2023].copy()

print(f"Train: {train_df.shape}  ({train_df['date'].dt.year.min()}–{train_df['date'].dt.year.max()})")
print(f"Val:   {val_df.shape}")
print(f"Test:  {test_df.shape}")


Train: (84738, 83)  (2018–2021)
Val:   (21170, 83)
Test:  (21170, 83)


## BUILDING THE TRANSFORMER MODEL

In [144]:

# ── Feature Groups ─────────────────────────────────────────────────────────────
WEATHER_TIME = [
    'tmax_k_pop', 'tmin_k_pop', 'trange_k',
    'cdd65_pop', 'hdd65_pop', 'cdd75_pop',
    'spfh_peak_kgkg_pop', 'wind_peak_ms_pop', 'dpd_k',
    'month', 'quarter', 'day_of_week', 'is_holiday',
    'baseline_mw_per_capita_2018', 'baseline_mw_monthly_2018'
]  # 15 features → 16 tokens with county

ROLLING_TIME = [
    'cdd65_pop_roll5', 'hdd65_pop_roll5',
    'tmax_k_pop_roll5_max', 'tmax_k_pop_roll7_mean', 'dpd_k_roll5',
    'month', 'quarter', 'day_of_week', 'is_holiday',
    'baseline_mw_per_capita_2018', 'baseline_mw_monthly_2018'
]  # 11 features → 12 tokens with county

GEO_NUMERIC = [
    'total_pop', 'per_cap_income',
    'baseline_mw_per_capita_2018', 'baseline_mw_monthly_2018'
]  # 4 numeric + county embedding

INFRA_TIME = [
    'cuml_count', 'cuml_sq_foot', 'cuml_utility_cap', 'cuml_dc_load', 'bev',
    'month', 'quarter', 'day_of_week', 'is_holiday',
    'baseline_mw_per_capita_2018', 'baseline_mw_monthly_2018'
]  # 11 features → 12 tokens with county

TARGET = 'max_elec_per_capita_log'
COUNTY_EMB_DIM = 8
EMBED_DIM = 32
NUM_HEADS = 8
CROSS_DIM = 64
DROPOUT = 0.1


In [145]:
# ── Dataset ────────────────────────────────────────────────────────────────────
class ClimateDataset(Dataset):
    def __init__(self, df, scalers=None, county_map=None, fit=False):

        w = df[WEATHER_TIME].values.astype(np.float32)
        r = df[ROLLING_TIME].values.astype(np.float32)
        g = df[GEO_NUMERIC].values.astype(np.float32)
        i = df[INFRA_TIME].values.astype(np.float32)

        if fit:
            self.scalers = {
                'w': StandardScaler().fit(w),
                'r': StandardScaler().fit(r),
                'g': StandardScaler().fit(g),
                'i': StandardScaler().fit(i),
            }
            self.county_map = {c: idx for idx, c in enumerate(df['county'].unique())}
        else:
            self.scalers    = scalers
            self.county_map = county_map

        self.w      = self.scalers['w'].transform(w)
        self.r      = self.scalers['r'].transform(r)
        self.g      = self.scalers['g'].transform(g)
        self.i      = self.scalers['i'].transform(i)
        self.county = df['county'].map(self.county_map).fillna(0).values.astype(np.int64)
        self.y      = df[TARGET].values.astype(np.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.w[idx]),
            torch.tensor(self.r[idx]),
            torch.tensor(self.g[idx]),
            torch.tensor(self.i[idx]),
            torch.tensor(self.county[idx]),
            torch.tensor(self.y[idx])
        )


In [146]:
# ── Attention Block ────────────────────────────────────────────────────────────
class AttentionBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        self.attn    = nn.MultiheadAttention(embed_dim, num_heads,
                                              dropout=dropout, batch_first=True)
        self.norm1   = nn.LayerNorm(embed_dim)
        self.ffn     = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 4, embed_dim),
        )
        self.norm2   = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)
        x = self.norm1(x + self.dropout(attn_out))
        x = self.norm2(x + self.dropout(self.ffn(x)))
        return x


In [147]:
# ── Stream Encoder ─────────────────────────────────────────────────────────────
class StreamEncoder(nn.Module):
    """Project features → embed_dim, prepend county token, 2 attention blocks, flatten"""
    def __init__(self, n_features, embed_dim=EMBED_DIM, num_heads=NUM_HEADS, dropout=DROPOUT):
        super().__init__()
        self.proj        = nn.Linear(1, embed_dim)
        self.county_proj = nn.Linear(COUNTY_EMB_DIM, embed_dim)
        self.block1      = AttentionBlock(embed_dim, num_heads, dropout)
        self.block2      = AttentionBlock(embed_dim, num_heads, dropout)
        self.out_dim     = (n_features + 1) * embed_dim  # +1 for county token

    def forward(self, x, county_emb):
        # x: (batch, n_features) → (batch, n_features, embed_dim)
        x = self.proj(x.unsqueeze(-1))
        # county: (batch, 8) → (batch, 1, embed_dim)
        c = self.county_proj(county_emb).unsqueeze(1)
        # prepend county token
        x = torch.cat([c, x], dim=1)   # (batch, n_features+1, embed_dim)
        x = self.block1(x)
        x = self.block2(x)
        return x.flatten(1)


In [148]:
# ── Full Model ─────────────────────────────────────────────────────────────────
class ClimateFEAT(nn.Module):
    def __init__(self, n_counties,
                 embed_dim=EMBED_DIM, num_heads=NUM_HEADS,
                 cross_dim=CROSS_DIM, dropout=DROPOUT):
        super().__init__()

        # County embedding — shared across all streams
        self.county_emb  = nn.Embedding(n_counties + 1, COUNTY_EMB_DIM)

        # Stream 1 — weather + time + baseline (15 features → 16 tokens)
        self.weather_enc = StreamEncoder(15, embed_dim, num_heads, dropout)

        # Stream 2 — rolling + time + baseline (11 features → 12 tokens)
        self.rolling_enc = StreamEncoder(11, embed_dim, num_heads, dropout)

        # Stream 3 — geography (county emb + 4 numeric → dense)
        self.geo_dense   = nn.Sequential(
            nn.Linear(COUNTY_EMB_DIM + 4, 32),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Stream 4 — infra + time + baseline (11 features → 12 tokens)
        self.infra_enc   = StreamEncoder(11, embed_dim, num_heads, dropout)

        # Phase 2 — cross-stream attention on streams 1+2+3
        self.w_to_cross  = nn.Linear(self.weather_enc.out_dim, cross_dim)  # 512 → 64
        self.r_to_cross  = nn.Linear(self.rolling_enc.out_dim, cross_dim)  # 384 → 64
        self.g_to_cross  = nn.Linear(32, cross_dim)                         # 32  → 64
        self.cross_attn  = AttentionBlock(cross_dim, num_heads=num_heads, dropout=dropout)
        # 3 tokens × cross_dim → 192

        # Final head — cross (192) + infra (384)
        head_input_dim = 3 * cross_dim + self.infra_enc.out_dim

        self.head = nn.Sequential(
            nn.Linear(head_input_dim, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 64),             nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 1)
        )
    def forward(self, w, r, g, i, county):
        county_emb = self.county_emb(county)                          # (batch, 8)

        # Phase 1 — within-stream, county token attends with each stream
        w_out = self.weather_enc(w, county_emb)                       # (batch, 512)
        r_out = self.rolling_enc(r, county_emb)                       # (batch, 384)
        g_out = self.geo_dense(torch.cat([g, county_emb], dim=1))     # (batch, 32)
        i_out = self.infra_enc(i, county_emb)                         # (batch, 384)

        # Phase 2 — cross-stream attention on weather + rolling + geo
        w_tok = self.w_to_cross(w_out).unsqueeze(1)                   # (batch, 1, 64)
        r_tok = self.r_to_cross(r_out).unsqueeze(1)                   # (batch, 1, 64)
        g_tok = self.g_to_cross(g_out).unsqueeze(1)                   # (batch, 1, 64)

        cross = torch.cat([w_tok, r_tok, g_tok], dim=1)               # (batch, 3, 64)
        cross = self.cross_attn(cross)
        cross = cross.flatten(1)                                       # (batch, 192)

        # Concat cross + infra → head
        out = self.head(torch.cat([cross, i_out], dim=1))
        return out.squeeze(-1)


In [149]:
# ── Datasets & Loaders ─────────────────────────────────────────────────────────
train_ds = ClimateDataset(train_df, fit=True)
val_ds   = ClimateDataset(val_df,
                          scalers=train_ds.scalers,
                          county_map=train_ds.county_map)

train_loader = DataLoader(train_ds, batch_size=512, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=512, shuffle=False, num_workers=2)



In [150]:
# ── Training ───────────────────────────────────────────────────────────────────
device     = torch.device('cuda')
n_counties = len(train_ds.county_map)
model      = ClimateFEAT(n_counties).to(device)
optimizer  = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-3)
criterion  = nn.MSELoss()
scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model params: {total_params:,}")
print(f"Training on:  {len(train_ds):,} samples")

best_val_loss  = float('inf')
patience_count = 0
PATIENCE       = 15

for epoch in range(150):
    model.train()
    train_losses = []
    for w, r, g, i, county, y in train_loader:
        w, r, g, i, county, y = [t.to(device) for t in [w, r, g, i, county, y]]
        optimizer.zero_grad()
        loss = criterion(model(w, r, g, i, county), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        train_losses.append(loss.item())

    model.eval()
    val_losses = []
    with torch.no_grad():
        for w, r, g, i, county, y in val_loader:
            w, r, g, i, county, y = [t.to(device) for t in [w, r, g, i, county, y]]
            val_losses.append(criterion(model(w, r, g, i, county), y).item())

    train_loss = np.mean(train_losses)
    val_loss   = np.mean(val_losses)
    scheduler.step(val_loss)

    if epoch % 5 == 0:
        print(f"Epoch {epoch:3d} | train {train_loss:.4f} | val {val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss  = val_loss
        torch.save(model.state_dict(), 'best_climate_feat.pt')
        patience_count = 0
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

model.load_state_dict(torch.load('best_climate_feat.pt'))
print(f"\nBest val loss: {best_val_loss:.4f}")

Model params: 351,961
Training on:  84,738 samples
Epoch   0 | train 3.1130 | val 0.0625
Epoch   5 | train 0.2848 | val 0.0516
Epoch  10 | train 0.2735 | val 0.0169
Epoch  15 | train 0.2479 | val 0.0455
Epoch  20 | train 0.2394 | val 0.0255
Epoch  25 | train 0.2203 | val 0.0303
Early stopping at epoch 29

Best val loss: 0.0159


In [151]:
model.eval()
all_preds = []

with torch.no_grad():
    for w, r, g, i, county, y in val_loader:
        w, r, g, i, county = [t.to(device) for t in [w, r, g, i, county]]
        preds = model(w, r, g, i, county)
        all_preds.append(preds.cpu().numpy())

preds_log = np.concatenate(all_preds)
val_df['preds_mwh_ft2'] = np.exp(preds_log) * val_df['total_pop']
val_df['actual_mwh']    = np.exp(val_df[TARGET]) * val_df['total_pop']

ft2_rmse = np.sqrt(mean_squared_error(val_df['actual_mwh'], val_df['preds_mwh_ft2']))
w       = val_df['total_pop'].values
wrmse   = np.sqrt(np.sum(w * (val_df['actual_mwh'] - val_df['preds_mwh_ft2'])**2) / np.sum(w))
wmean   = np.average(val_df['actual_mwh'], weights=w)
pop_wtd = 100 * wrmse / wmean

print(f"ClimateFEAT Transformer v2:")
print(f"  RMSE MWh:        {ft2_rmse:,.0f}")
print(f"  Pop-wtd RMSE %:  {pop_wtd:.1f}%")

# LA monthly bias
val_df['residual_mwh'] = val_df['actual_mwh'] - val_df['preds_mwh_ft2']
la = val_df[val_df['county'] == 'Los Angeles'].copy()
la['month'] = la['date'].dt.month
print("\nLA Monthly Bias:")
print(la.groupby('month').agg(
    bias_mwh=('residual_mwh', 'mean'),
    rmse_mwh=('residual_mwh', lambda x: np.sqrt((x**2).mean()))
).round(1).to_string())

ClimateFEAT Transformer v2:
  RMSE MWh:        180
  Pop-wtd RMSE %:  15.6%

LA Monthly Bias:
       bias_mwh  rmse_mwh
month                    
1         852.6    1091.1
2         651.1    1053.0
3         438.9    1128.0
4         316.6    1104.1
5         125.9     893.6
6        -232.2     620.7
7       -1563.5    1705.5
8       -1313.5    1478.9
9       -1263.8    1704.0
10         46.2     760.2
11        415.2     932.2
12         87.0     725.8


In [152]:
#!pip install mlflow

In [153]:
import mlflow
import joblib
import os
from datetime import datetime

run_ts        = datetime.now().strftime('%Y%m%d_%H%M')
model_version = 'v1'
model_name    = f'climatefeat_transformer_{model_version}_{run_ts}'

MODEL_DIR = '/content/drive/My Drive/210_capstone/MLFlow/models'
PRED_DIR  = '/content/drive/My Drive/210_capstone/MLFlow/predictions'
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(PRED_DIR,  exist_ok=True)

mlflow.set_tracking_uri(f'sqlite:///{MODEL_DIR}/mlflow.db')
mlflow.set_experiment('climate-feat-transformer')

with mlflow.start_run(run_name=model_name):

    # ── Params ────────────────────────────────────────────────────────────
    mlflow.log_param('model_version',   model_version)
    mlflow.log_param('embed_dim',       32)
    mlflow.log_param('num_heads',       4)
    mlflow.log_param('dropout',         0.1)
    mlflow.log_param('batch_size',      512)
    mlflow.log_param('lr',              3e-4)
    mlflow.log_param('weight_decay',    1e-3)
    mlflow.log_param('n_weather_feats', len(WEATHER_TIME))
    mlflow.log_param('n_rolling_feats', len(ROLLING_TIME))
    mlflow.log_param('n_geo_feats',     len(GEO_NUMERIC))
    mlflow.log_param('n_infra_feats',   len(INFRA_TIME))
    mlflow.log_param('best_epoch',      epoch)

    # ── Metrics ───────────────────────────────────────────────────────────
    mlflow.log_metric('best_val_loss',   best_val_loss)
    mlflow.log_metric('rmse_mwh',        ft2_rmse)
    mlflow.log_metric('pop_wtd_rmse_pct', pop_wtd)

    # ── Save model ────────────────────────────────────────────────────────
    torch.save(model.state_dict(), f'{MODEL_DIR}/{model_name}.pt')
    mlflow.log_artifact(f'{MODEL_DIR}/{model_name}.pt')

    # ── Save scalers ──────────────────────────────────────────────────────
    joblib.dump(train_ds.scalers,    f'{MODEL_DIR}/{model_name}_scalers.pkl')
    joblib.dump(train_ds.county_map, f'{MODEL_DIR}/{model_name}_county_map.pkl')
    mlflow.log_artifact(f'{MODEL_DIR}/{model_name}_scalers.pkl')
    mlflow.log_artifact(f'{MODEL_DIR}/{model_name}_county_map.pkl')

    # ── Save predictions ──────────────────────────────────────────────────
    pred_df = val_df[['county', 'date', 'total_pop', 'actual_mwh',
                       'preds_mwh_ft2', 'residual_mwh']].copy()
    pred_df.to_parquet(f'{PRED_DIR}/{model_name}_preds_val.parquet', index=False)
    mlflow.log_artifact(f'{PRED_DIR}/{model_name}_preds_val.parquet')

    print(f'MLflow run logged — climate-feat-transformer / {model_name}')
    print(f'  best_val_loss:    {best_val_loss:.4f}')
    print(f'  rmse_mwh:         {ft2_rmse:,.0f}')
    print(f'  pop_wtd_rmse_pct: {pop_wtd:.1f}%')

MLflow run logged — climate-feat-transformer / climatefeat_transformer_v1_20260302_2255
  best_val_loss:    0.0159
  rmse_mwh:         180
  pop_wtd_rmse_pct: 15.6%
